## How to Summarize Scientific Papers Using the BART Model with Hugging Face Transformers

Based on [this article on KDnuggets](https://www.kdnuggets.com/how-to-summarize-scientific-papers-bart-hugging-face-transformers)

### Preparation

In [ ]:
## !pip install transformers 

In [1]:
## !pip install PyTorch
# !pip install torch 

### Scientific Paper Summarization with BART


In [ ]:
import fitz  

def extract_paper_text(pdf_path):
    text = ""
    doc = fitz.open(pdf_path)
    for page in doc:
        text += page.get_text()
    return text
pdf_path = "attention_is_all_you_need.pdf"
cleaned_text = extract_paper_text(pdf_path)

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

def summarize_text(text, model, tokenizer, max_chunk_size=1024):
    chunks = [text[i:i+max_chunk_size] for i in range(0, len(text), max_chunk_size)]
    summaries = []
    for chunk in chunks:
        inputs = tokenizer(chunk, max_length=max_chunk_size, return_tensors="pt", truncation=True)
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=200,
            min_length=50,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )
        summaries.append(tokenizer.decode(summary_ids[0], skip_special_tokens=True))
    return " ".join(summaries)

summary = summarize_text(cleaned_text, model, tokenizer)

In [ ]:
def hierarchical_summarization(text, model, tokenizer, max_chunk_size=1024):
    first_level_summary = summarize_text(text, model, tokenizer, max_chunk_size)
   
    inputs = tokenizer(first_level_summary, max_length=max_chunk_size, return_tensors="pt", truncation=True)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=200,
        min_length=50,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )
    final_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
   
    return final_summary

final_summary = hierarchical_summarization(cleaned_text, model, tokenizer)